# K-Nearest Neighbors (KNN)
## Machine Learning — Universidad de Talca

---

## Índice

1. [Configuración del Entorno](#sec1)
2. [Introducción y Fundamentos](#sec2)
3. [Métricas de Similitud y Distancia](#sec3)
4. [El Hiperparámetro K y Fronteras de Decisión](#sec4)
5. [Estructuras de Datos y Complejidad Algorítmica](#sec5)
6. [Preprocesamiento Crítico](#sec6)
7. [Ejercicio Integrador](#sec7)

---

**Convención de color:**
- <span style="color:#0056b3; font-weight:bold">■ Azul → Pregrado</span>
- <span style="background-color:#fff9c4; padding:2px 8px; border-radius:3px; font-weight:bold">■ Amarillo → Posgrado (omitir en clase de pregrado)</span>

<a id="sec1"></a>

## 1. Configuración del Entorno

In [ ]:
import os

# Verificación de archivos locales de respaldo
data_files = ['data/penguins.csv']
missing = [f for f in data_files if not os.path.exists(f)]
if missing:
    print("⚠️  Archivos locales no encontrados:")
    for f in missing:
        print(f"   - {f}")
    print("   Se intentará descarga online.")
    print("   Para descargar manualmente:")
    print("   !curl -s https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv -o data/penguins.csv")
else:
    print("✓ Archivos locales disponibles.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons, load_digits, load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (9, 5)
print("✓ Librerías cargadas.")

In [ ]:
# ── Penguins ──────────────────────────────────────────────────────
try:
    import seaborn as sns
    df_penguins = sns.load_dataset('penguins').dropna()
    print("✓ Penguins: descargado online")
except Exception:
    df_penguins = pd.read_csv('data/penguins.csv').dropna()
    print("✓ Penguins: cargado desde data/penguins.csv")

# ── make_moons (sintético) ────────────────────────────────────────
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
print("✓ make_moons: generado localmente")

# ── load_digits ───────────────────────────────────────────────────
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
print(f"✓ load_digits: {X_digits.shape[0]} muestras, {X_digits.shape[1]} dimensiones")

# ── load_wine ─────────────────────────────────────────────────────
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
print(f"✓ load_wine: {X_wine.shape[0]} muestras, {X_wine.shape[1]} features")

<a id="sec2"></a>

---

## 2. Introducción y Fundamentos

<div style="color: #0056b3;">

### ¿Qué es KNN?

KNN es un algoritmo de aprendizaje supervisado con tres características clave:

- **No paramétrico:** no asume una forma funcional para los datos
- **Basado en instancias (Lazy Learning):** no entrena un modelo; memoriza los datos
- **Instancia de consulta:** para clasificar un punto nuevo, busca sus K vecinos más cercanos y vota

> *"Dime con quién te juntas y te diré quién eres."*

</div>

<div style="color: #0056b3;">

### Fases del Algoritmo

| # | Fase | Descripción |
| --- | --- | --- |
| 1 | Almacenamiento | Guardar todos los puntos $(X_{train}, y_{train})$ |
| 2 | Consulta | Recibir un punto nuevo $x_q$ |
| 3 | Distancias | Calcular $d(x_q, x_i)$ para todo $x_i \in X_{train}$ |
| 4 | Selección | Ordenar y tomar los $K$ más cercanos |
| 5 | Votación | Clase mayoritaria (clasificación) o promedio (regresión) |

</div>

In [ ]:
# Intuición visual: cómo cambia la frontera de decisión según K
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
h = 0.02
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
cmap_bg = ListedColormap(['#AED6F1', '#A9DFBF'])
cmap_pt = ListedColormap(['#2980B9', '#1E8449'])

for ax, k in zip(axes, [1, 5, 20]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_moons, y_moons)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=cmap_pt,
               s=20, edgecolors='k', linewidths=0.3)
    ax.set_title(f'K = {k}', fontsize=14)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Frontera de decisión KNN — make_moons', fontsize=13)
plt.tight_layout()
plt.show()

---

## [POSGRADO] Lazy Learning vs. Modelos Paramétricos

<div style="background-color: #fff9c4; padding: 10px; border-radius: 5px; color: #000;">

### Contraste de Complejidad Computacional

| | KNN (Basado en Instancias) | Regresión Logística (Paramétrico) |
| --- | --- | --- |
| **Entrenamiento** | $O(1)$ — solo almacenar | $O(N \cdot d \cdot \text{iter})$ — optimizar pesos |
| **Inferencia** | $O(N \cdot d)$ — buscar vecinos | $O(d)$ — producto punto |
| **Memoria** | $O(N \cdot d)$ — guardar todo | $O(d)$ — solo pesos |
| **Online learning** | Trivial (agregar punto) | Requiere re-entrenamiento |

**Consecuencia práctica:** para $N = 10^6$ y $d = 100$, clasificar **un solo punto** con fuerza bruta requiere $10^8$ operaciones.

</div>

---
<!-- fin bloque posgrado -->

<a id="sec3"></a>

## 3. Métricas de Similitud y Distancia

<div style="color: #0056b3;">

### Familia de Distancias de Minkowski

$$d_p(x, y) = \left( \sum_{i=1}^{d} |x_i - y_i|^p \right)^{1/p}$$

| $p$ | Nombre | Geometría |
| --- | --- | --- |
| $p = 1$ | Manhattan | Cuadrícula urbana |
| $p = 2$ | Euclidiana | Línea recta |
| $p \to \infty$ | Chebyshev | Máxima diferencia en cualquier eje |

**Clave:** KNN es sensible a la escala de las variables. Una variable en rango $[0,1000]$ domina sobre una en $[0,1]$.

</div>

In [ ]:
# Bolas unitarias para distintas normas p
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
configs = [
    (1,  'Manhattan ($p=1$)',       '#E74C3C'),
    (2,  'Euclidiana ($p=2$)',       '#2980B9'),
    (10, 'Chebyshev ($p\\to\\infty$)', '#27AE60'),
]
angles = np.linspace(0, 2 * np.pi, 1000)
x_unit = np.cos(angles)
y_unit = np.sin(angles)

for ax, (p, title, color) in zip(axes, configs):
    norms = (np.abs(x_unit)**p + np.abs(y_unit)**p) ** (1/p)
    xb, yb = x_unit / norms, y_unit / norms
    ax.fill(xb, yb, alpha=0.3, color=color)
    ax.plot(xb, yb, color=color, lw=2)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Bolas unitarias para distintas normas $p$', fontsize=12)
plt.tight_layout()
plt.show()

---

## [POSGRADO] Distancia de Mahalanobis y Metric Learning

<div style="background-color: #fff9c4; padding: 10px; border-radius: 5px; color: #000;">

### Distancia de Mahalanobis

La distancia Euclidiana ignora correlaciones entre variables. La Mahalanobis las corrige:

$$d_M(x, y) = \sqrt{(x - y)^T \, \Sigma^{-1} \, (x - y)}$$

**Interpretación geométrica:** transforma los datos para tener varianza unitaria y correlación cero, *luego* aplica distancia Euclidiana.

Cuando $\Sigma = I$: $d_M = d_E$ (Euclidiana es caso especial).

**Metric Learning:** en lugar de usar $\Sigma$ calculada, se puede *aprender* una matriz $M$ que minimiza distancias intra-clase y maximiza distancias inter-clase.

</div>

In [ ]:
# Demostración: Euclidiana vs. Mahalanobis en datos correlacionados
def mahalanobis_distance(x, y, VI):
    diff = x - y
    return np.sqrt(diff @ VI @ diff)

np.random.seed(42)
cov = [[1, 0.9], [0.9, 1]]  # alta correlación
X_corr = np.random.multivariate_normal([0, 0], cov, 200)
VI = np.linalg.inv(np.cov(X_corr.T))

ref = np.array([0.0, 0.0])
p1 = np.array([1.5,  1.4])  # sigue la correlación
p2 = np.array([1.5, -1.4])  # va contra la correlación

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax in axes:
    ax.scatter(X_corr[:, 0], X_corr[:, 1], alpha=0.3, s=10, color='gray')
    ax.scatter(*ref, color='black', s=120, zorder=5, label='Referencia')
    ax.scatter(*p1, color='#E74C3C', s=120, zorder=5, label='p1 (sigue correlación)')
    ax.scatter(*p2, color='#2980B9', s=120, zorder=5, label='p2 (rompe correlación)')
    ax.set_aspect('equal'); ax.legend(fontsize=8)

de1, de2 = np.linalg.norm(ref-p1), np.linalg.norm(ref-p2)
dm1, dm2 = mahalanobis_distance(ref,p1,VI), mahalanobis_distance(ref,p2,VI)

axes[0].set_title(f'Euclidiana\np1={de1:.2f}  p2={de2:.2f}  (iguales)', fontsize=11)
axes[1].set_title(f'Mahalanobis\np1={dm1:.2f}  p2={dm2:.2f}  (p2 más lejos)', fontsize=11)
plt.suptitle('Mahalanobis reconoce la estructura de correlación', fontsize=12)
plt.tight_layout()
plt.show()

---
<!-- fin bloque posgrado -->

<a id="sec4"></a>

## 4. El Hiperparámetro K y Fronteras de Decisión

<div style="color: #0056b3;">

### El Compromiso Sesgo–Varianza

| K pequeño | K grande |
| --- | --- |
| Fronteras irregulares | Fronteras suaves |
| Alta varianza → sobreajuste | Alto sesgo → subajuste |
| Sensible al ruido | Ignora estructura local |
| K=1: celdas de Voronoi | K=N: predice siempre clase mayoritaria |

**Regla práctica inicial:** probar $K = \sqrt{N}$ como punto de partida.  
**Selección formal:** validación cruzada (cross-validation).

</div>

In [ ]:
# Widget interactivo: slider de K sobre make_moons
# Funciona en VS Code y Colab con ipywidgets instalado

def plot_knn_boundary(K=5):
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X_moons, y_moons)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    train_acc = accuracy_score(y_moons, knn.predict(X_moons))
    cv_acc = cross_val_score(knn, X_moons, y_moons, cv=5).mean()
    cmap_bg = ListedColormap(['#AED6F1', '#A9DFBF'])
    cmap_pt = ListedColormap(['#2980B9', '#1E8449'])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=cmap_pt,
               s=25, edgecolors='k', linewidths=0.3)
    ax.set_title(f'K={K}  |  Train Acc: {train_acc:.2%}  |  CV Acc (5-fold): {cv_acc:.2%}',
                 fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as widgets
    widgets.interact(plot_knn_boundary,
                     K=widgets.IntSlider(value=5, min=1, max=30, step=1,
                                         description='K:',
                                         style={'description_width': 'initial'}))
except ImportError:
    print("⚠️ ipywidgets no disponible — versión estática:")
    h = 0.03
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    cmap_bg = ListedColormap(['#AED6F1', '#A9DFBF'])
    cmap_pt = ListedColormap(['#2980B9', '#1E8449'])
    for ax, k in zip(axes, [1, 3, 5, 10, 20]):
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_moons, y_moons)
        Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
        ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=cmap_pt,
                   s=15, edgecolors='k', lw=0.2)
        cv = cross_val_score(knn, X_moons, y_moons, cv=5).mean()
        ax.set_title(f'K={k}\nCV={cv:.2%}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle('Fronteras de decisión para distintos K', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# Selección de K óptimo por validación cruzada
k_range = range(1, 31)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                              X_moons, y_moons, cv=5).mean()
             for k in k_range]
best_k = list(k_range)[np.argmax(cv_scores)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_range, cv_scores, 'o-', color='#2980B9', lw=2, ms=5)
ax.axvline(best_k, color='#E74C3C', linestyle='--', label=f'K óptimo = {best_k}')
ax.set_xlabel('K', fontsize=12)
ax.set_ylabel('Accuracy (CV 5-fold)', fontsize=12)
ax.set_title('Selección de K por Validación Cruzada — make_moons', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"K óptimo: {best_k}  |  CV Accuracy: {max(cv_scores):.2%}")

---

## [POSGRADO] Distance Weighting y Regresión Local (LOESS)

<div style="background-color: #fff9c4; padding: 10px; border-radius: 5px; color: #000;">

### KNN con Ponderación por Distancia

En lugar de voto uniforme, ponderar cada vecino inversamente a su distancia:

$$\hat{y} = \frac{\sum_{i \in \mathcal{N}_K(x)} w_i \cdot y_i}{\sum_{i \in \mathcal{N}_K(x)} w_i}, \qquad w_i = \frac{1}{d(x, x_i)^2}$$

**Ventaja:** vecinos más cercanos tienen mayor influencia. Reduce el efecto de outliers en la frontera.

### Regresión Local (LOESS/LOWESS)

Generalización para regresión: ajusta una regresión ponderada local usando kernel tricúbico:

$$w_i = \left(1 - \left(\frac{d(x, x_i)}{d_{\max}}\right)^3\right)^3$$

Produce estimaciones suaves sin asumir forma global.

</div>

In [ ]:
# Comparación: voto uniforme vs. ponderado por distancia (K=7)
h = 0.02
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
cmap_bg = ListedColormap(['#AED6F1', '#A9DFBF'])
cmap_pt = ListedColormap(['#2980B9', '#1E8449'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (weights, title) in zip(axes, [('uniform', 'Voto Uniforme'),
                                        ('distance', 'Ponderado por Distancia')]):
    knn = KNeighborsClassifier(n_neighbors=7, weights=weights)
    knn.fit(X_moons, y_moons)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cv = cross_val_score(knn, X_moons, y_moons, cv=5).mean()
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=cmap_pt,
               s=20, edgecolors='k', lw=0.3)
    ax.set_title(f'{title} (K=7)\nCV Accuracy: {cv:.2%}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

---
<!-- fin bloque posgrado -->

<a id="sec5"></a>

## 5. Estructuras de Datos y Complejidad Algorítmica

<div style="color: #0056b3;">

### Clasificación de Dígitos con KNN

`load_digits`: 1,797 imágenes de dígitos escritos a mano, 8×8 píxeles = **64 dimensiones**.

**Observación clave:** KNN no aprende nada. Para clasificar un nuevo dígito, compara su vector de 64 píxeles contra los 1,797 del entrenamiento.

Con $N = 1{,}797$ y $d = 64$: cada predicción requiere $1{,}797 \times 64 = 115{,}008$ operaciones.

</div>

In [ ]:
# Visualizar muestras del dataset
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(str(digits.target[i]), fontsize=10)
    ax.axis('off')
plt.suptitle('load_digits — 20 muestras (8×8 píxeles)', fontsize=12)
plt.tight_layout()
plt.show()

# KNN básico en digits
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_ds = sc.fit_transform(X_tr_d)
X_te_ds = sc.transform(X_te_d)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr_ds, y_tr_d)
acc = accuracy_score(y_te_d, knn.predict(X_te_ds))
print(f"KNN (K=5, StandardScaler) en load_digits → Test Accuracy: {acc:.2%}")

---

## [POSGRADO] KD-Trees, Ball Trees y Análisis de Complejidad

<div style="background-color: #fff9c4; padding: 10px; border-radius: 5px; color: #000;">

### Complejidad Computacional

| Algoritmo | Construcción | Consulta | Mejor escenario |
| --- | --- | --- | --- |
| Fuerza Bruta | $O(1)$ | $O(N \cdot d)$ | Siempre exacto |
| KD-Tree | $O(N \cdot d \log N)$ | $O(d \log N)$ | $d \leq 20$ |
| Ball Tree | $O(N \cdot d \log N)$ | $O(d \log N)$ | Alta dimensión |
| LSH | $O(N)$ | $O(1)$ aprox. | Búsqueda aproximada |

**KD-Tree:** divide el espacio con hiperplanos paralelos a los ejes. Falla en alta dimensión: el radio de búsqueda debe crecer para encontrar K vecinos.

**Ball Tree:** divide usando hiperesferas. Más robusto que KD-Tree para $d > 20$ e invariante a rotaciones.

</div>

In [ ]:
import time

# Benchmark: Fuerza bruta vs. KD-Tree vs. Ball Tree en load_digits
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42)
sc_b = StandardScaler()
X_tr_b = sc_b.fit_transform(X_tr_b)
X_te_b = sc_b.transform(X_te_b)

print(f"{'Algoritmo':<12} {'Fit (ms)':>10} {'Predict (ms)':>14} {'Accuracy':>10}")
print("-" * 52)

results_bench = {}
for algo in ['brute', 'kd_tree', 'ball_tree']:
    knn = KNeighborsClassifier(n_neighbors=5, algorithm=algo)
    t0 = time.perf_counter()
    knn.fit(X_tr_b, y_tr_b)
    t_fit = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    preds = knn.predict(X_te_b)
    t_pred = (time.perf_counter() - t0) * 1000
    acc = accuracy_score(y_te_b, preds)
    print(f"{algo:<12} {t_fit:>10.2f} {t_pred:>14.2f} {acc:>10.2%}")
    results_bench[algo] = (t_pred, acc)

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(7, 4))
algos = list(results_bench.keys())
times = [results_bench[a][0] for a in algos]
colors = ['#E74C3C', '#2980B9', '#27AE60']
bars = ax.bar(algos, times, color=colors, edgecolor='black', alpha=0.85)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{t:.1f} ms', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Tiempo de predicción (ms)', fontsize=12)
ax.set_title('Benchmark: Fuerza Bruta vs. Árboles de Partición Espacial\nload_digits', fontsize=11)
plt.tight_layout()
plt.show()

---
<!-- fin bloque posgrado -->

<a id="sec6"></a>

## 6. Preprocesamiento Crítico

<div style="color: #0056b3;">

### ¿Por qué escalar antes de KNN?

KNN usa distancias. Si las variables tienen escalas distintas, la de mayor rango domina.

**Ejemplo — Palmer Penguins:**
- `body_mass_g` ∈ [2700, 6300] g
- `bill_length_mm` ∈ [32, 60] mm

Sin escalar, la masa corporal determina casi toda la distancia.

| Escalado | Fórmula | Resultado |
| --- | --- | --- |
| StandardScaler | $z = (x - \mu) / \sigma$ | media 0, varianza 1 |
| MinMaxScaler | $x' = (x - x_{min}) / (x_{max} - x_{min})$ | rango [0, 1] |

</div>

In [ ]:
# Impacto del escalado en KNN — Palmer Penguins
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X_p = df_penguins[features].values
y_p = LabelEncoder().fit_transform(df_penguins['species'])

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_p, y_p, test_size=0.2, random_state=42, stratify=y_p)

results_scale = {}
for name, scaler in [('Sin escalar', None),
                      ('StandardScaler', StandardScaler()),
                      ('MinMaxScaler', MinMaxScaler())]:
    Xtr = scaler.fit_transform(X_tr_p) if scaler else X_tr_p
    Xte = scaler.transform(X_te_p)    if scaler else X_te_p
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(Xtr, y_tr_p)
    results_scale[name] = accuracy_score(y_te_p, knn.predict(Xte))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(results_scale.keys(), results_scale.values(),
              color=['#E74C3C', '#2980B9', '#27AE60'], edgecolor='black', alpha=0.85)
ax.set_ylabel('Accuracy (Test)', fontsize=12)
ax.set_title('Impacto del Escalado en KNN — Palmer Penguins (K=5)', fontsize=12)
ax.set_ylim(0, 1.12)
for bar, val in zip(bars, results_scale.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2%}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---

## [POSGRADO] Maldición de la Dimensionalidad

<div style="background-color: #fff9c4; padding: 10px; border-radius: 5px; color: #000;">

### Curse of Dimensionality

En dimensión alta, los puntos se vuelven equidistantes. Para $N$ puntos uniformes en $[0,1]^d$:

$$\frac{d_{\max} - d_{\min}}{d_{\min}} \xrightarrow{d \to \infty} 0$$

**Consecuencia para KNN:** los K vecinos más cercanos dejan de ser "cercanos" en sentido geométrico. La noción de vecindad local colapsa.

**Solución práctica:** reducir dimensionalidad con PCA antes de aplicar KNN.

Demostraremos esto con `load_digits` aplicando PCA a distintos números de componentes.

</div>

In [ ]:
# Demostración: accuracy KNN en load_digits según dimensión (PCA)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_digits, y_digits, test_size=0.25, random_state=42)
sc_c = StandardScaler()
X_tr_c = sc_c.fit_transform(X_tr_c)
X_te_c = sc_c.transform(X_te_c)

n_components_list = [2, 5, 10, 20, 30, 40, 50, 64]
accs_pca = []
for n in n_components_list:
    pca = PCA(n_components=n, random_state=42)
    Xtr_pca = pca.fit_transform(X_tr_c)
    Xte_pca = pca.transform(X_te_c)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(Xtr_pca, y_tr_c)
    accs_pca.append(accuracy_score(y_te_c, knn.predict(Xte_pca)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_components_list, accs_pca, 'o-', color='#8E44AD', lw=2, ms=7)
ax.set_xlabel('Componentes PCA (dimensión)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('KNN en load_digits: Accuracy vs. Dimensionalidad', fontsize=12)
ax.grid(True, alpha=0.3)
for x, y_val in zip(n_components_list, accs_pca):
    ax.annotate(f'{y_val:.2%}', (x, y_val),
                textcoords='offset points', xytext=(0, 9),
                ha='center', fontsize=8)
plt.tight_layout()
plt.show()

best_n = n_components_list[np.argmax(accs_pca)]
print(f"Mejor dimensión: {best_n} componentes → Accuracy: {max(accs_pca):.2%}")

---
<!-- fin bloque posgrado -->

<a id="sec7"></a>

## 7. Ejercicio Integrador

<div style="color: #0056b3;">

### Dataset: Wine Recognition

`load_wine` contiene análisis químicos de vinos de 3 cultivares italianos:

- **178 muestras**, **13 features** (alcohol, flavanoids, color intensity, etc.)
- **3 clases** (cultivar 0, 1, 2) — disponible offline desde sklearn

### Objetivo

Construir el mejor clasificador KNN posible.

1. Comparar accuracy con y sin escalado
2. Encontrar el K óptimo por validación cruzada
3. Interpretar los resultados

</div>

In [ ]:
# Ejercicio Pregrado: pipeline completo sobre Wine
print("=" * 55)
print("EJERCICIO INTEGRADOR — Wine Dataset")
print("=" * 55)

X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine)

# [1] Sin escalar
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_tr_w, y_tr_w)
acc_raw = accuracy_score(y_te_w, knn_raw.predict(X_te_w))
print(f"\n[1] Sin escalar       — K=5  — Test Acc: {acc_raw:.2%}")

# [2] Con StandardScaler
sc_w = StandardScaler()
X_tr_ws = sc_w.fit_transform(X_tr_w)
X_te_ws = sc_w.transform(X_te_w)
knn_sc = KNeighborsClassifier(n_neighbors=5)
knn_sc.fit(X_tr_ws, y_tr_w)
acc_sc = accuracy_score(y_te_w, knn_sc.predict(X_te_ws))
print(f"[2] StandardScaler    — K=5  — Test Acc: {acc_sc:.2%}")

# [3] K óptimo por CV
k_range_w = range(1, 31)
cv_scores_w = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                                X_tr_ws, y_tr_w, cv=5).mean()
               for k in k_range_w]
best_k_w = list(k_range_w)[np.argmax(cv_scores_w)]
knn_best_w = KNeighborsClassifier(n_neighbors=best_k_w)
knn_best_w.fit(X_tr_ws, y_tr_w)
acc_best_w = accuracy_score(y_te_w, knn_best_w.predict(X_te_ws))
print(f"[3] StandardScaler    — K={best_k_w:<2} — Test Acc: {acc_best_w:.2%}  ← K óptimo")

print(f"\nMejora por escalado:   {acc_sc - acc_raw:+.2%}")
print(f"Mejora por K óptimo:   {acc_best_w - acc_sc:+.2%}")

---

## [POSGRADO] Ejercicio con Distancia de Mahalanobis

In [ ]:
# KNN con Distancia de Mahalanobis en Wine
VI_wine = np.linalg.inv(np.cov(X_tr_ws.T))

knn_maha = KNeighborsClassifier(
    n_neighbors=best_k_w,
    metric='mahalanobis',
    metric_params={'VI': VI_wine}
)
knn_maha.fit(X_tr_ws, y_tr_w)
acc_maha = accuracy_score(y_te_w, knn_maha.predict(X_te_ws))

print("=" * 55)
print(f"Comparación final — Wine (K={best_k_w})")
print("=" * 55)
print(f"Euclidiana (StandardScaler): {acc_best_w:.2%}")
print(f"Mahalanobis:                 {acc_maha:.2%}")
print(f"Diferencia:                  {acc_maha - acc_best_w:+.2%}")
print("\nNota: con StandardScaler los datos ya tienen varianza unitaria,")
print("por lo que Mahalanobis ≈ Euclidiana cuando las correlaciones son bajas.")

---
<!-- fin bloque posgrado -->